# 모델 종합 검증 (RAG 평가 관점)

이 노트북은 우리 프로젝트(bokji-agent)에서 후보로 정한 LLM이 실제 정책 데이터에서도
신뢰할 수 있는지 검증한다. `13_lm_evaluation/01_llm_eval.ipynb`에서 배운 개념 중,
우리 프로젝트(RAG 챗봇)에 실제로 해당하는 두 가지를 우리 데이터로 적용한다.

| RAG 평가 관점 | 이 노트북에서 확인하는 것 |
|---|---|
| 형식 준수 (Structured Output) | LLM이 정해진 JSON 스키마를 얼마나 잘 지키는가 |
| 근거 충실성 (Faithfulness) | LLM이 원문에 없는 내용(조작된 사실)을 만들어냈을 때, 검증 단계가 이를 잡아내는가 |

검증 정확도는 분류 문제로 본다 (01_llm_eval.ipynb의 Precision/Recall 개념과 동일):
- **Recall(재현율)**: 실제로 조작된 답 중, 검증 단계가 '불일치'로 제대로 잡아낸 비율
  → 이게 낮으면 챗봇이 지어낸 정보를 사용자에게 그대로 보여줄 위험이 커진다 (가장 위험한 실패)
- **Specificity(특이도)**: 실제로 정상인 답 중, 검증 단계가 '일치'로 제대로 판정한 비율
  → 이게 낮으면 정상 답변도 자꾸 '확인 필요'로 처리해서 사용자 경험이 나빠진다

## 실행 전제
- vLLM으로 모델을 로컬(RunPod A40 등)에 띄워둔 상태여야 한다: `vllm serve <model> --max-model-len 16384 --port 8000`
- 테스트 데이터는 `data/samples/subsidy_documents_sample.jsonl`의 실제 정책 문서를 사용한다
  (표본이 5건뿐이라 방향성 확인용이다. 전체 10,957건 데이터를 구할 수 있으면 `DATA_PATH`만 바꿔서 표본을 늘릴 수 있다)

In [ ]:
# 설정 — vLLM 서버 주소와 평가할 모델을 여기서 바꾼다.
from openai import OpenAI

BASE_URL = "http://localhost:8000/v1"
MODEL = "skt/A.X-4.0-Light"  # 비교하려면: Qwen/Qwen3.5-9B, Bllossom/llama-3.2-Korean-Bllossom-3B
DATA_PATH = "../../data/samples/subsidy_documents_sample.jsonl"

client = OpenAI(base_url=BASE_URL, api_key="not-needed")

## 1) 테스트 데이터 준비

실제 정책 문서에서 LLM에게 넘길 원문(source_text)을 만든다. `근거법령` 섹션은
N5(claim_plan.py)에서 이미 별도로 처리하는 영역이라 이 요약 생성에서는 제외한다.

In [ ]:
import json

SECTION_ORDER = ["목적", "지원대상", "선정기준", "지원내용", "신청방법", "신청기한"]


def load_test_documents(path: str) -> list[dict]:
    docs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            docs.append(json.loads(line))
    return docs


def build_source_text(doc: dict) -> str:
    parts = [f"정책명: {doc['title']}"]
    sections_by_heading = {tuple(s["heading_path"]): s["content"] for s in doc.get("sections", [])}
    for heading in SECTION_ORDER:
        content = sections_by_heading.get((heading,))
        if content:
            parts.append(f"{heading}: {content}")
    return "\n\n".join(parts)


test_documents = load_test_documents(DATA_PATH)
print(f"테스트 문서 {len(test_documents)}건 로드됨")
for d in test_documents:
    print("-", d["title"])

## 2) 목적 1 — 형식 일정하게 (Structured Output)

각 정책마다 LLM에게 요약을 요청하고, 정해진 스키마(정책명/지원형태/최종결과_문구/배지_상태)를
그대로 지켰는지 확인한다.

In [ ]:
gen_schema = {
    "type": "object",
    "properties": {
        "정책명": {"type": "string"},
        "지원형태": {"type": "string"},
        "최종결과_문구": {"type": "string"},
        "배지_상태": {"type": "string", "enum": ["자격_충족", "확인_필요"]},
    },
    "required": ["정책명", "지원형태", "최종결과_문구", "배지_상태"],
}


def generate_structured(source_text: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": source_text + "\n\n위 정책 정보를 정리해서 답해줘."}],
        response_format={"type": "json_schema", "json_schema": {"name": "policy_summary", "schema": gen_schema}},
    )
    return response.choices[0].message.content


def schema_is_valid(generated_str: str) -> bool:
    try:
        data = json.loads(generated_str)
    except json.JSONDecodeError:
        return False
    required = ["정책명", "지원형태", "최종결과_문구", "배지_상태"]
    return all(k in data and isinstance(data[k], str) and data[k] for k in required)

## 3) 목적 2 — 내용 원문 일치 확인 (LLM-as-Judge)

생성된 요약이 원문에 없는 내용을 담고 있을 때, 검증 호출이 이를 잡아내는지 확인한다.
`inject_fabrication()`은 생성된 요약을 일부러 조작한다 (숫자가 있으면 2배로 부풀리고,
없으면 원문에 절대 있을 수 없는 가짜 조건을 덧붙인다).

In [ ]:
import re
import copy

verify_schema = {
    "type": "object",
    "properties": {
        "일치_여부": {"type": "string", "enum": ["일치", "불일치"]},
        "불일치_근거": {"type": "string"},
    },
    "required": ["일치_여부", "불일치_근거"],
}


def inject_fabrication(generated_str: str) -> str:
    data = json.loads(generated_str)
    corrupted = copy.deepcopy(data)
    text = corrupted.get("최종결과_문구", "")
    numbers = re.findall(r"\d[\d,]*", text)
    if numbers:
        original = numbers[0]
        fake_val = int(original.replace(",", "")) * 2
        corrupted["최종결과_문구"] = text.replace(original, f"{fake_val:,}", 1)
    else:
        corrupted["최종결과_문구"] = text + " (단, 신청자의 생년월일이 홀수인 경우에만 해당됩니다.)"
    return json.dumps(corrupted, ensure_ascii=False)


def verify_against_source(source_text: str, generated_str: str) -> dict:
    prompt = f"""아래 [원문]과 [생성된 문장]을 비교해줘.
[생성된 문장]에 [원문]에 없는 정보(숫자, 조건, 사실)가 하나라도 들어가 있으면 \"불일치\"로 판정하고 그 부분을 구체적으로 적어줘.
[원문]에 있는 내용만으로 이루어져 있으면 \"일치\"로 판정해줘.

[원문]
{source_text}

[생성된 문장]
{generated_str}
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_schema", "json_schema": {"name": "verify_result", "schema": verify_schema}},
    )
    return json.loads(response.choices[0].message.content)

## 4) 전체 평가 루프

문서마다: (1) 요약 생성 + 스키마 검사, (2) 정상 생성물 검증(기대값: 일치),
(3) 조작된 생성물 검증(기대값: 불일치)을 수행하고 결과를 모은다.

In [ ]:
results = []

for doc in test_documents:
    source_text = build_source_text(doc)
    generated = generate_structured(source_text)
    schema_ok = schema_is_valid(generated)

    clean_verdict = verify_against_source(source_text, generated)
    corrupted = inject_fabrication(generated) if schema_ok else generated
    corrupted_verdict = verify_against_source(source_text, corrupted)

    results.append({
        "title": doc["title"],
        "schema_ok": schema_ok,
        "generated": generated,
        "clean_verdict": clean_verdict.get("일치_여부"),
        "clean_correct": clean_verdict.get("일치_여부") == "일치",
        "corrupted_verdict": corrupted_verdict.get("일치_여부"),
        "corrupted_caught": corrupted_verdict.get("일치_여부") == "불일치",
        "corrupted_reason": corrupted_verdict.get("불일치_근거"),
    })
    print(f"완료: {doc['title']}")

## 5) 결과 집계

In [ ]:
n = len(results)
schema_pass = sum(r["schema_ok"] for r in results)
clean_correct = sum(r["clean_correct"] for r in results)
corrupted_caught = sum(r["corrupted_caught"] for r in results)

print(f"모델: {MODEL}")
print(f"표본 수: {n}건")
print(f"스키마 준수율: {schema_pass}/{n} ({schema_pass/n:.0%})")
print(f"정상 케이스 '일치' 정답률 (specificity): {clean_correct}/{n} ({clean_correct/n:.0%})")
print(f"조작 케이스 '불일치' 탐지율 (recall): {corrupted_caught}/{n} ({corrupted_caught/n:.0%})  <- 제일 중요, 낮으면 환각을 못 거른다는 뜻")
print()
print("문서별 상세:")
for r in results:
    status = "OK" if (r["schema_ok"] and r["clean_correct"] and r["corrupted_caught"]) else "확인 필요"
    print(f"  [{status}] {r['title']} | 스키마={r['schema_ok']} | 정상판정={r['clean_verdict']} | 조작판정={r['corrupted_verdict']}")

## 결과 해석 및 다음 단계

- **스키마 준수율**이 100%가 아니면: 모델을 바꾸거나 프롬프트를 조정해야 한다
- **조작 케이스 탐지율(recall)**이 낮으면: 이 모델을 검증 단계(LLM-as-judge)에 그대로 쓰면 안 된다 —
  규칙 기반 대조([document_verification.py](../../src/rag_chatbot/graph/nodes/document_verification.py))처럼
  더 확실한 방식과 병행하거나 대체해야 한다
- 표본이 5건뿐이라 이 결과는 방향성 확인용이다. `MODEL`을 바꿔가며(Qwen3.5-9B, Bllossom-3B 등)
  같은 셀들을 다시 실행하면 모델별 비교표를 만들 수 있다 (앞서 한 실험과 동일한 방식)